In [5]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Locate the repository root
ROOT = Path.cwd()

if not (ROOT / "src" / "shared_preprocessing.py").exists():
    ROOT = ROOT.parent

if not (ROOT / "src" / "shared_preprocessing.py").exists():
    raise FileNotFoundError("Cannot locate src/shared_preprocessing.py")

sys.path.insert(0, str(ROOT))

# Import the shared preprocessing functions
from src.shared_preprocessing import (
    load_raw_data,
    prepare_features_target,
    get_split,
    build_preprocessor,
    make_cv,
    RANDOM_STATE,
    TEST_SIZE,
    N_SPLITS,
    PRIMARY_SCORING,
)

print("Shared preprocessing module imported successfully.")
print("Repository root located:", ROOT.is_dir())
print("Random state:", RANDOM_STATE)
print("Test size:", TEST_SIZE)
print("Cross-validation folds:", N_SPLITS)
print("Primary scoring metric:", PRIMARY_SCORING)

Shared preprocessing module imported successfully.
Repository root located: True
Random state: 42
Test size: 0.2
Cross-validation folds: 5
Primary scoring metric: f1


In [2]:
# Load the dataset and generate the stratified train/test split
raw_df = load_raw_data()

X_train, X_test, y_train, y_test = get_split()

# Dataset and split dimensions
print("Original dataset:", raw_df.shape)
print("Training predictors:", X_train.shape)
print("Testing predictors:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

# Verify the split
assert len(X_train) + len(X_test) == len(raw_df)
assert len(X_train) == 36168
assert len(X_test) == 9043
assert set(y_train.unique()) == {0, 1}
assert set(y_test.unique()) == {0, 1}

# Display target distribution in both sets
target_distribution = pd.DataFrame({
    "Training": y_train.value_counts().sort_index(),
    "Testing": y_test.value_counts().sort_index(),
})

target_distribution.index = ["No subscription (0)", "Subscribed (1)"]
target_distribution["Training (%)"] = (
    target_distribution["Training"] / len(y_train) * 100
).round(2)
target_distribution["Testing (%)"] = (
    target_distribution["Testing"] / len(y_test) * 100
).round(2)

display(target_distribution)

# Verify feature transformations
assert "duration" not in X_train.columns
assert "campaign" not in X_train.columns
assert "campaign_prior_contacts" in X_train.columns
assert "pdays" not in X_train.columns
assert "days_since_previous_contact" in X_train.columns
assert "previous" in X_train.columns

assert (X_train["campaign_prior_contacts"] >= 0).all()
assert X_train["days_since_previous_contact"].isna().any()
assert "unknown" in X_train["poutcome"].values

# Verify feature types
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

print("\nNumeric features:", len(numeric_features), numeric_features)
print("Categorical features:", len(categorical_features), categorical_features)

assert len(numeric_features) == 6
assert len(categorical_features) == 9

print("\nAll dataset split and feature transformation checks passed.")

Original dataset: (45211, 17)
Training predictors: (36168, 15)
Testing predictors: (9043, 15)
Training target: (36168,)
Testing target: (9043,)


,Training,Testing,Training (%),Testing (%)
No subscription (0),31937,7985,88.3,88.3
Subscribed (1),4231,1058,11.7,11.7



Numeric features: 6 ['age', 'balance', 'day', 'previous', 'campaign_prior_contacts', 'days_since_previous_contact']
Categorical features: 9 ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

All dataset split and feature transformation checks passed.


In [3]:
from scipy import sparse

# Build the shared preprocessing transformer
preprocessor = build_preprocessor(X_train)

# Fit using training data only, then transform both sets
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Inspect transformed dimensions
print("Original training dimensions:", X_train.shape)
print("Transformed training dimensions:", X_train_transformed.shape)
print("Original testing dimensions:", X_test.shape)
print("Transformed testing dimensions:", X_test_transformed.shape)

print("\nOutput format:",
      "Sparse matrix" if sparse.issparse(X_train_transformed)
      else "Dense array")

# Validate the transformations
assert X_train_transformed.shape[0] == len(X_train)
assert X_test_transformed.shape[0] == len(X_test)
assert X_train_transformed.shape[1] == X_test_transformed.shape[1]

# Confirm that the transformed values are finite
if sparse.issparse(X_train_transformed):
    assert np.isfinite(X_train_transformed.data).all()
    assert np.isfinite(X_test_transformed.data).all()
else:
    assert np.isfinite(X_train_transformed).all()
    assert np.isfinite(X_test_transformed).all()

print("\nPreprocessing transformation checks passed.")

Original training dimensions: (36168, 15)
Transformed training dimensions: (36168, 50)
Original testing dimensions: (9043, 15)
Transformed testing dimensions: (9043, 50)

Output format: Dense array

Preprocessing transformation checks passed.


In [4]:
# Validate the cross-validation configuration
cv = make_cv()

fold_results = []
validation_indices = []

for fold_number, (train_indices, validation_fold_indices) in enumerate(
    cv.split(X_train, y_train), start=1
):
    y_fold_train = y_train.iloc[train_indices]
    y_fold_validation = y_train.iloc[validation_fold_indices]

    validation_indices.extend(validation_fold_indices)

    fold_results.append({
        "Fold": fold_number,
        "Training rows": len(train_indices),
        "Validation rows": len(validation_fold_indices),
        "Training positive rate (%)": round(y_fold_train.mean() * 100, 2),
        "Validation positive rate (%)": round(
            y_fold_validation.mean() * 100, 2
        ),
    })

fold_summary = pd.DataFrame(fold_results)
display(fold_summary)

# Confirm the five-fold split
assert len(fold_results) == N_SPLITS
assert len(validation_indices) == len(X_train)
assert len(set(validation_indices)) == len(X_train)
assert sorted(validation_indices) == list(range(len(X_train)))

print("Number of folds:", N_SPLITS)
print("Primary scoring metric:", PRIMARY_SCORING)
print("Random state:", RANDOM_STATE)
print("\nCross-validation checks passed.")

,Fold,Training rows,Validation rows,Training positive rate (%),Validation positive rate (%)
0,1,28934,7234,11.7,11.69
1,2,28934,7234,11.7,11.69
2,3,28934,7234,11.7,11.71
3,4,28935,7233,11.7,11.70
4,5,28935,7233,11.7,11.70


Number of folds: 5
Primary scoring metric: f1
Random state: 42

Cross-validation checks passed.


## 5. Validation Summary

The shared preprocessing module passed the following checks:

- Module imports and configuration.
- Dataset loading and stratified 80:20 train/test split.
- Target encoding and feature engineering.
- Confirmation that post-call `duration` is excluded.
- Numeric and categorical feature identification.
- Imputation, standardisation and one-hot encoding.
- Consistent transformed dimensions across training and test sets.
- Five-fold stratified cross-validation configuration.

The final feature matrix contains 50 encoded features. The training and test sets retain the original positive-class proportion of 11.7%.

All validation assertions passed.

**Note:** The preprocessing transformation in Section 3 was performed for diagnostic validation only. Final model experiments must include the preprocessing transformer within each model pipeline so that fitting occurs separately within each cross-validation fold.